# Fine-tuning pair visualization

This notebook shows the query, positive, and negative images stored in a Lynx or CzechLynx fine-tuning index.

It is intentionally diagnostic:

- image-only inspection works without a model or GPU;
- cached RDD/LoMa features can be inspected;
- fresh RDD/LightGlue or LoMa scores are optional;
- suspicious identities, self-matches, missing images, and missing cache entries are reported.

Run it from the benchmark repository or set `RDD_BENCHMARK_ROOT` explicitly.

In [ ]:
from pathlib import Path
import json
import math
import os
import random
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw
from IPython.display import Markdown, display

# Find the repository even when Jupyter was started from notebook/.
_REPO_CANDIDATES = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (p for p in _REPO_CANDIDATES if (p / 'scripts').is_dir() and (p / 'README.md').exists()),
    Path(os.environ.get('RDD_BENCHMARK_ROOT', '/home/kargin/Projects/repositories/rdd-parallel-benchmark')),
)
FINETUNING_ROOT = Path(
    os.environ.get('LYNX_FINETUNING_ROOT', '/home/kargin/Projects/repositories/lynx-finetuning')
)
for import_root in (REPO_ROOT, FINETUNING_ROOT):
    if str(import_root) not in sys.path:
        sys.path.insert(0, str(import_root))

# Current CzechLynx legacy defaults. Override these values or use environment variables.
DATASET_ROOT = Path(os.environ.get(
    'DATASET_ROOT',
    '/shared/sets/datasets/vision/czechlynx/CzechLynx_processed_time_closed',
))
INDEX_ROOT = Path(os.environ.get(
    'WILDLIFE_INDEX_ROOT',
    '/home/kargin/Projects/repositories/rdd-parallel-benchmark/outputs/czechlynx-time-closed/legacy',
))
MINING_BACKEND = os.environ.get('PAIR_MINING_BACKEND', 'rdd').lower()
if MINING_BACKEND not in {'rdd', 'loma'}:
    raise ValueError(f'Unsupported mining backend: {MINING_BACKEND}; expected rdd or loma')
MINING_INDEX_ROOT = INDEX_ROOT / MINING_BACKEND
_PAIR_INDEX_OVERRIDE = os.environ.get('PAIR_INDEX')
RDD_CACHE_ROOT = Path(os.environ.get(
    'RDD_CACHE_ROOT',
    '/shared/sets/datasets/vision/czechlynx/checkpoints/czechlynx-time-closed/rdd-cache',
))
LOMA_CACHE_ROOT = Path(os.environ.get(
    'LOMA_CACHE_ROOT',
    '/shared/sets/datasets/vision/czechlynx/checkpoints/czechlynx-time-closed/loma-b-cache',
))
MINING_CACHE_ROOT = RDD_CACHE_ROOT if MINING_BACKEND == 'rdd' else LOMA_CACHE_ROOT
RDD_PRETRAINED = Path(os.environ.get(
    'RDD_PRETRAINED',
    '/home/kargin/Projects/repositories/lynx-finetuning/rdd/weights/RDD_lg-v2.pth',
))
LOMA_PRETRAINED = Path(os.environ.get(
    'LOMA_PRETRAINED',
    '/shared/sets/datasets/confidential/lynx/checkpoints/loma/loma_B.pt',
))
RDD_CHECKPOINT = os.environ.get('RDD_CHECKPOINT', '')
LOMA_CHECKPOINT = os.environ.get('LOMA_CHECKPOINT', '')

SPLIT = os.environ.get('PAIR_SPLIT', 'train')
if SPLIT not in {'train', 'val', 'test'}:
    raise ValueError(f'Unsupported pair split: {SPLIT}')

def _default_index_for_split(split):
    if _PAIR_INDEX_OVERRIDE:
        return Path(_PAIR_INDEX_OVERRIDE)
    backend_root = MINING_INDEX_ROOT
    if MINING_BACKEND == 'loma' or backend_root.exists() or 'PAIR_MINING_BACKEND' in os.environ:
        return backend_root / f'strong-matches_{split}_combined.json'
    # Historical shared index fallback for the original Lynx/CzechLynx layout.
    return INDEX_ROOT / f'strong-matches_{split}_combined.json'

INDEX_PATH = _default_index_for_split(SPLIT)
SEED = int(os.environ.get('PAIR_SEED', '0'))
N_QUERIES = int(os.environ.get('N_QUERIES', '4'))
N_POSITIVES = int(os.environ.get('N_POSITIVES', '2'))
N_NEGATIVES = int(os.environ.get('N_NEGATIVES', '2'))
SELECTION_MODE = os.environ.get('PAIR_SELECTION', 'mixed')  # random, strongest, weakest, mixed
SCORE_BACKEND = os.environ.get('PAIR_SCORE_BACKEND', 'none')  # none, rdd, loma
DEVICE = os.environ.get('PAIR_DEVICE', 'cuda' if __import__('torch').cuda.is_available() else 'cpu')
SHOW_OVERLAYS = os.environ.get('SHOW_MATCH_OVERLAYS', '0') == '1'
MAX_OVERLAY_LINES = int(os.environ.get('MAX_OVERLAY_LINES', '40'))
OUTPUT_ROOT = REPO_ROOT / 'notebook' / 'outputs' / 'pair_visualizations'

print(f'Repository: {REPO_ROOT}')
print(f'Dataset:    {DATASET_ROOT}')
print(f'Mining backend: {MINING_BACKEND}')
print(f'Mining index root: {MINING_INDEX_ROOT}')
print(f'Index:      {INDEX_PATH}')
print(f'Mining cache: {MINING_CACHE_ROOT}')
print(f'Device:     {DEVICE}; backend: {SCORE_BACKEND}; seed: {SEED}')

In [ ]:
def _candidate_frame(item):
    if isinstance(item, str):
        return item, None
    if not isinstance(item, dict):
        raise TypeError(f'Unsupported candidate type: {type(item)!r}')
    frame = item.get('frame') or item.get('path') or item.get('image')
    if not frame:
        raise ValueError(f'Candidate has no frame/path/image field: {item}')
    score = item.get('score', item.get('selection_score'))
    return str(frame), None if score is None else float(score)


def _normalise_entry(entry):
    if not isinstance(entry, dict) or 'query_frame' not in entry:
        raise ValueError(f'Invalid index entry: {entry}')
    positives = [_candidate_frame(item) for item in entry.get('positives', [])]
    negatives = [_candidate_frame(item) for item in entry.get('negatives', [])]
    return {
        'query_frame': str(entry['query_frame']),
        'positives': [{'frame': path, 'mining_score': score} for path, score in positives],
        'negatives': [{'frame': path, 'mining_score': score} for path, score in negatives],
        'query_frame_index': entry.get('query_frame_index'),
    }


def _entries_from_raw(raw):
    if isinstance(raw, list):
        return raw
    if not isinstance(raw, dict):
        raise ValueError('Index JSON must contain a list or object')
    for key in ('entries', 'selected_frames', 'frames', 'data'):
        value = raw.get(key)
        if isinstance(value, list):
            return value
    if 'query_frame' in raw:
        return [raw]
    raise ValueError('Could not find entries, selected_frames, frames, or data in index JSON')


def load_index(index_path):
    raw = json.loads(Path(index_path).read_text())
    entries = [_normalise_entry(entry) for entry in _entries_from_raw(raw)]
    if not entries:
        raise ValueError(f'No usable entries found in {index_path}')
    return entries


def resolve_frame(frame, dataset_root=DATASET_ROOT):
    path = Path(frame)
    if path.is_absolute():
        return path
    candidate = dataset_root / path
    if candidate.exists():
        return candidate
    # Helpful when an index was written with a repository-relative prefix.
    candidate = REPO_ROOT / path
    if candidate.exists():
        return candidate
    return dataset_root / path


def path_metadata(frame):
    parts = Path(frame).parts
    result = {
        'split': parts[0] if len(parts) > 0 else None,
        'identity': parts[1] if len(parts) > 1 else None,
        'source': parts[2] if len(parts) > 2 else None,
        'encounter_or_sequence': parts[3] if len(parts) > 3 else None,
        'filename': parts[-1] if parts else None,
    }
    result['collection'] = (
        f"{result['identity']}/{result['source']}"
        if result['identity'] and result['source'] else None
    )
    return result


def _same_path(a, b):
    try:
        return resolve_frame(a).resolve() == resolve_frame(b).resolve()
    except OSError:
        return str(Path(a)) == str(Path(b))


def validate_entries(entries, dataset_root=DATASET_ROOT, max_report=20):
    issues = []
    checked_pairs = 0
    for entry_index, entry in enumerate(entries):
        query = entry['query_frame']
        query_meta = path_metadata(query)
        if not resolve_frame(query, dataset_root).exists():
            issues.append(('missing_query', entry_index, query))
        for kind in ('positives', 'negatives'):
            for candidate in entry[kind]:
                frame = candidate['frame']
                meta = path_metadata(frame)
                checked_pairs += 1
                if not resolve_frame(frame, dataset_root).exists():
                    issues.append((f'missing_{kind[:-1]}', entry_index, frame))
                if kind == 'positives' and meta['identity'] != query_meta['identity']:
                    issues.append(('positive_identity_mismatch', entry_index, query, frame))
                if kind == 'negatives' and meta['identity'] == query_meta['identity']:
                    issues.append(('negative_identity_match', entry_index, query, frame))
                if _same_path(query, frame):
                    issues.append(('exact_query_self_match', entry_index, query, frame))
    print(f'Entries: {len(entries):,}; candidate pairs: {checked_pairs:,}')
    print(f'Issues: {len(issues):,}')
    for issue in issues[:max_report]:
        print('  ', issue)
    if len(issues) > max_report:
        print(f'  ... {len(issues) - max_report:,} more issues')
    return issues


def print_entry_summary(entry):
    query_meta = path_metadata(entry['query_frame'])
    print('Query:', entry['query_frame'])
    print('  identity:', query_meta['identity'], '| source:', query_meta['source'],
          '| encounter/sequence:', query_meta['encounter_or_sequence'])
    print('  positives:', len(entry['positives']), '| negatives:', len(entry['negatives']))

In [ ]:
def cache_path(frame, cache_root):
    return Path(cache_root) / Path(frame).with_suffix('.npz')


def load_cached_feature(frame, cache_root):
    path = cache_path(frame, cache_root)
    if not path.exists():
        raise FileNotFoundError(path)
    with np.load(path, allow_pickle=False) as data:
        fields = set(data.files)
        if {'keypoints', 'descriptors', 'scores', 'image_size'}.issubset(fields):
            keypoints = np.asarray(data['keypoints'])
            descriptors = np.asarray(data['descriptors'])
            scores = np.asarray(data['scores'])
            image_size = np.asarray(data['image_size'])
        elif {'k', 'd', 'hw'}.issubset(fields):
            keypoints = np.asarray(data['k'])
            descriptors = np.asarray(data['d'])
            scores = np.ones(keypoints.shape[0], dtype=np.float32)
            image_size = np.asarray(data['hw'])
        else:
            raise ValueError(f'Unsupported cache schema in {path}: {sorted(fields)}')
    return {
        'path': path,
        'keypoints': keypoints,
        'descriptors': descriptors,
        'scores': scores,
        'image_size': tuple(int(x) for x in image_size),
    }


def cache_report(entries, cache_root, max_report=20):
    missing = []
    incompatible = []
    frames = {entry['query_frame'] for entry in entries}
    for entry in entries:
        frames.update(item['frame'] for item in entry['positives'])
        frames.update(item['frame'] for item in entry['negatives'])
    for frame in sorted(frames):
        try:
            load_cached_feature(frame, cache_root)
        except FileNotFoundError:
            missing.append(frame)
        except (ValueError, OSError) as exc:
            incompatible.append((frame, str(exc)))
    print(f'Cache: {cache_root}')
    print(f'Unique referenced frames: {len(frames):,}')
    print(f'Missing: {len(missing):,}; incompatible: {len(incompatible):,}')
    for item in missing[:max_report]:
        print('  missing:', item)
    for item in incompatible[:max_report]:
        print('  incompatible:', item)
    return missing, incompatible


def feature_info(frame, cache_root):
    feat = load_cached_feature(frame, cache_root)
    return {
        'frame': frame,
        'cache_file': str(feat['path']),
        'keypoints': int(feat['keypoints'].shape[0]),
        'descriptor_shape': tuple(feat['descriptors'].shape),
        'image_size_hw': feat['image_size'],
    }


def show_feature_info(entry, cache_root):
    frames = [entry['query_frame']]
    frames += [item['frame'] for item in entry['positives'][:2]]
    frames += [item['frame'] for item in entry['negatives'][:2]]
    rows = []
    for frame in frames:
        try:
            rows.append(feature_info(frame, cache_root))
        except Exception as exc:
            rows.append({'frame': frame, 'error': str(exc)})
    for row in rows:
        print(row)

In [ ]:
def _image_or_placeholder(frame, dataset_root=DATASET_ROOT, image_size=None):
    path = resolve_frame(frame, dataset_root)
    if not path.exists():
        image = Image.new('RGB', (640, 360), 'white')
        draw = ImageDraw.Draw(image)
        draw.text((20, 20), f'MISSING IMAGE\n{frame}', fill='red')
        return image
    image = Image.open(path).convert('RGB')
    if image_size is not None:
        width, height = int(image_size[1]), int(image_size[0])
        image = image.resize((width, height), Image.Resampling.BILINEAR)
    return image


def _select_candidates(candidates, count, mode, rng):
    if not candidates or count <= 0:
        return []
    count = min(count, len(candidates))
    scored = all(item.get('mining_score') is not None for item in candidates)
    if mode == 'strongest' and scored:
        return sorted(candidates, key=lambda item: item['mining_score'], reverse=True)[:count]
    if mode == 'weakest' and scored:
        return sorted(candidates, key=lambda item: item['mining_score'])[:count]
    if mode == 'mixed':
        if not scored:
            chosen = list(candidates)
            rng.shuffle(chosen)
            return chosen[:count]
        ordered = sorted(candidates, key=lambda item: item['mining_score'], reverse=True)
        chosen = ordered[:1]
        if count > 1:
            chosen += ordered[-1:]
        remaining = [item for item in candidates if item not in chosen]
        rng.shuffle(remaining)
        return (chosen + remaining)[:count]
    chosen = list(candidates)
    rng.shuffle(chosen)
    return chosen[:count]


def _pair_title(kind, frame, query_frame, score=None):
    meta = path_metadata(frame)
    query_meta = path_metadata(query_frame)
    label = f'{kind}: {meta["identity"]}'
    if meta['source']:
        label += f'/{meta["source"]}'
    if score is not None:
        label += f'\nscore={score:.4f}'
    if meta['identity'] == query_meta['identity']:
        label += '\nSAME IDENTITY'
    return label


def _score_or_mining(scorer, query_frame, candidate):
    if scorer is not None:
        try:
            return scorer.score(query_frame, candidate['frame'])
        except Exception as exc:
            return {'error': str(exc)}
    if candidate.get('mining_score') is not None:
        return {'score': candidate['mining_score'], 'matches': None, 'source': 'stored mining score'}
    return {}


def show_pair_grid(
    entry,
    n_positives=N_POSITIVES,
    n_negatives=N_NEGATIVES,
    selection_mode=SELECTION_MODE,
    seed=SEED,
    dataset_root=DATASET_ROOT,
    scorer=None,
    save_path=None,
):
    rng = random.Random(seed)
    selected_pos = _select_candidates(entry['positives'], n_positives, selection_mode, rng)
    selected_neg = _select_candidates(entry['negatives'], n_negatives, selection_mode, rng)
    selected = [('positive', item) for item in selected_pos] + [('negative', item) for item in selected_neg]
    if not selected:
        raise ValueError('Entry contains no candidates')
    columns = len(selected) + 1
    fig, axes = plt.subplots(1, columns, figsize=(4.3 * columns, 5.3), squeeze=False)
    axes = axes[0]
    query = entry['query_frame']
    axes[0].imshow(_image_or_placeholder(query, dataset_root))
    qmeta = path_metadata(query)
    axes[0].set_title(
        f'QUERY\n{qmeta["identity"]}/{qmeta["source"] or "source"}\n'
        f'{qmeta["encounter_or_sequence"] or ""}',
        fontsize=10,
    )
    axes[0].axis('off')
    results = []
    for axis, (kind, candidate) in zip(axes[1:], selected):
        frame = candidate['frame']
        result = _score_or_mining(scorer, query, candidate)
        score = result.get('score')
        title = _pair_title(kind.upper(), frame, query, score)
        if result.get('matches') is not None:
            title += f'\nmatches={result["matches"]}'
        if result.get('error'):
            title += '\nSCORE ERROR'
        axis.imshow(_image_or_placeholder(frame, dataset_root))
        axis.set_title(title, fontsize=9)
        axis.axis('off')
        results.append({
            'kind': kind,
            'query_frame': query,
            'candidate_frame': frame,
            **path_metadata(frame),
            **result,
            'self_match': _same_path(query, frame),
        })
    fig.suptitle('Fine-tuning pair inspection', fontsize=14)
    fig.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=160, bbox_inches='tight')
        print('Saved:', save_path)
    display(fig)
    plt.close(fig)
    return results


def show_examples(
    entries,
    n_queries=N_QUERIES,
    **grid_kwargs,
):
    rng = random.Random(grid_kwargs.pop('seed', SEED))
    chosen = list(entries)
    rng.shuffle(chosen)
    chosen = chosen[:min(n_queries, len(chosen))]
    all_results = []
    for number, entry in enumerate(chosen):
        print(f'Example {number + 1}/{len(chosen)}')
        print_entry_summary(entry)
        all_results.append(show_pair_grid(entry, seed=SEED + number, **grid_kwargs))
    return all_results

In [ ]:
import torch

class PairScorer:
    """Optional fresh scorer over cached features.

    The image-only notebook path does not import RDD or LoMa. Instantiate this
    class only when the relevant environment, checkpoint, and cache are ready.
    """

    def __init__(self, backend, model, cache_root, device):
        self.backend = backend
        self.model = model
        self.cache_root = Path(cache_root)
        self.device = torch.device(device)
        self._feature_cache = {}

    def feature(self, frame):
        if frame not in self._feature_cache:
            self._feature_cache[frame] = load_cached_feature(frame, self.cache_root)
        return self._feature_cache[frame]

    def score(self, query_frame, candidate_frame):
        if self.backend == 'rdd':
            from scripts.lynx_benchmark import FrameFeat, score_pair_lightglue
            q = self.feature(query_frame)
            g = self.feature(candidate_frame)
            qf = FrameFeat(q['keypoints'], q['descriptors'], q['scores'], np.asarray(q['image_size']))
            gf = FrameFeat(g['keypoints'], g['descriptors'], g['scores'], np.asarray(g['image_size']))
            score, matches = score_pair_lightglue(self.model, qf, gf, self.device)
            return {'score': float(score), 'matches': int(matches), 'source': 'fresh RDD/LightGlue'}

        if self.backend == 'loma':
            from scripts.loma_backend import score_batch_loma
            q = self.feature(query_frame)
            g = self.feature(candidate_frame)
            qf = type('Feature', (), q)()
            gf = type('Feature', (), g)()
            score = score_batch_loma(self.model, [qf], [gf], self.device)[0, 0]
            return {'score': float(score), 'matches': None, 'source': 'fresh LoMa'}

        raise ValueError(f'Unsupported backend: {self.backend}')


def _load_state(path):
    path = Path(path)
    if path.is_dir():
        for name in ('model.safetensors', 'matcher.safetensors', 'weights.pth'):
            candidate = path / name
            if candidate.exists():
                path = candidate
                break
    if path.suffix == '.safetensors':
        from safetensors.torch import load_file
        state = load_file(str(path), device='cpu')
    else:
        state = torch.load(str(path), map_location='cpu')
    if isinstance(state, dict):
        for key in ('state_dict', 'model', 'weights', 'matcher'):
            if isinstance(state.get(key), dict):
                state = state[key]
                break
    return {key.removeprefix('module.'): value for key, value in state.items()
            if isinstance(value, torch.Tensor)}


def _overlay_state(model, checkpoint):
    if not checkpoint:
        return model
    state = _load_state(checkpoint)
    model_state = model.state_dict()
    matched = {key: value for key, value in state.items() if key in model_state}
    if not matched:
        raise ValueError(f'No checkpoint keys matched the selected {type(model).__name__}')
    missing, unexpected = model.load_state_dict(matched, strict=False)
    print(f'Loaded {len(matched):,} checkpoint tensors; missing={len(missing):,}, unexpected={len(unexpected):,}')
    return model


def build_pair_scorer(backend, checkpoint, cache_root, device):
    if backend in (None, '', 'none'):
        return None
    device = torch.device(device)
    if backend == 'rdd':
        from contrastive_finetuning.models import build_masked_lg
        model = build_masked_lg(device, weights=str(RDD_PRETRAINED))
        model = _overlay_state(model, checkpoint)
        return PairScorer('rdd', model, cache_root, device)
    if backend == 'loma':
        from scripts.loma_backend import build_loma
        model = build_loma(device, Path(LOMA_PRETRAINED), variant='loma-b')
        model = _overlay_state(model, checkpoint)
        return PairScorer('loma', model, cache_root, device)
    raise ValueError('backend must be none, rdd, or loma')


def _raw_correspondences(scorer, query_frame, candidate_frame):
    """Return actual matched keypoint coordinates for optional overlay plots."""
    q = scorer.feature(query_frame)
    g = scorer.feature(candidate_frame)
    device = scorer.device
    qk = torch.from_numpy(q['keypoints']).float().unsqueeze(0).to(device)
    qd = torch.from_numpy(q['descriptors']).float().unsqueeze(0).to(device)
    gk = torch.from_numpy(g['keypoints']).float().unsqueeze(0).to(device)
    gd = torch.from_numpy(g['descriptors']).float().unsqueeze(0).to(device)

    with torch.inference_mode():
        if scorer.backend == 'rdd':
            size0 = torch.tensor(
                [q['image_size'][1], q['image_size'][0]], device=device
            ).unsqueeze(0)
            size1 = torch.tensor(
                [g['image_size'][1], g['image_size'][0]], device=device
            ).unsqueeze(0)
            pred = scorer.model({
                'image0': {'keypoints': qk, 'descriptors': qd, 'image_size': size0},
                'image1': {'keypoints': gk, 'descriptors': gd, 'image_size': size1},
            })
            matches0 = pred['matches0'][0].detach().cpu().numpy()
            scores0 = pred['matching_scores0'][0].detach().cpu().numpy()
        else:
            from loma.loma import filter_matches
            scores = scorer.model(qk, gk, qd, gd)['scores']
            base = scorer.model.module if hasattr(scorer.model, 'module') else scorer.model
            matches0, _, scores0, _ = filter_matches(
                scores, base.cfg.filter_threshold
            )
            matches0 = matches0[0].detach().cpu().numpy()
            scores0 = scores0[0].detach().cpu().numpy()

    # Keep only actual, in-range correspondences. This also protects the
    # overlay from malformed/padded matcher output.
    valid = (
        (matches0 >= 0)
        & (matches0 < g['keypoints'].shape[0])
        & np.isfinite(scores0)
    )
    return (
        np.asarray(q['keypoints'])[valid],
        np.asarray(g['keypoints'])[matches0[valid].astype(np.int64)],
        np.asarray(scores0)[valid],
    )


def _keypoints_to_pixels(points, image_size):
    """Convert either pixel or normalized keypoints to pixel coordinates."""
    points = np.asarray(points, dtype=np.float32).reshape(-1, 2).copy()
    if not len(points):
        return points

    height, width = (int(image_size[0]), int(image_size[1]))
    finite = np.isfinite(points)
    if not finite.all():
        points[~finite] = 0

    # Some feature backends expose [0, 1] or [-1, 1] coordinates. The
    # benchmark caches normally contain pixel coordinates, so leave those
    # unchanged unless their range clearly identifies a normalized format.
    minimum = float(points.min())
    maximum = float(points.max())
    if minimum >= -1.01 and maximum <= 1.01:
        if minimum < -0.01:
            points[:, 0] = (points[:, 0] + 1.0) * 0.5 * max(width - 1, 1)
            points[:, 1] = (points[:, 1] + 1.0) * 0.5 * max(height - 1, 1)
        else:
            points[:, 0] *= max(width - 1, 1)
            points[:, 1] *= max(height - 1, 1)
    return points


def show_match_overlay(scorer, query_frame, candidate_frame, dataset_root=DATASET_ROOT,
                       max_lines=MAX_OVERLAY_LINES, save_path=None):
    """Draw actual matcher correspondences on one side-by-side image canvas."""
    if scorer is None:
        raise ValueError('Build a PairScorer before requesting overlays')

    q_feature = scorer.feature(query_frame)
    g_feature = scorer.feature(candidate_frame)
    q_points, g_points, match_scores = _raw_correspondences(
        scorer, query_frame, candidate_frame
    )

    # Feature coordinates are defined in the resized cache image space, not
    # necessarily in the original image space. Display the same resized images.
    q_image = np.asarray(_image_or_placeholder(
        query_frame, dataset_root, image_size=q_feature['image_size']
    ))
    g_image = np.asarray(_image_or_placeholder(
        candidate_frame, dataset_root, image_size=g_feature['image_size']
    ))
    qh, qw = q_image.shape[:2]
    gh, gw = g_image.shape[:2]
    q_points = _keypoints_to_pixels(q_points, (qh, qw))
    g_points = _keypoints_to_pixels(g_points, (gh, gw))

    canvas = np.zeros((max(qh, gh), qw + gw, 3), dtype=np.uint8)
    canvas[:qh, :qw] = q_image[:, :, :3]
    canvas[:gh, qw:qw + gw] = g_image[:, :, :3]

    order = np.argsort(match_scores)[::-1][:max(0, int(max_lines))]
    finite_scores = np.asarray(match_scores)[order]
    finite_scores = finite_scores[np.isfinite(finite_scores)]
    if len(finite_scores) and float(finite_scores.max()) > float(finite_scores.min()):
        color_norm = plt.Normalize(float(finite_scores.min()), float(finite_scores.max()))
    else:
        color_norm = plt.Normalize(0.0, 1.0)

    fig_width = max(12.0, (qw + gw) / 80.0)
    fig_height = max(6.0, max(qh, gh) / 80.0)
    fig, axis = plt.subplots(figsize=(fig_width, fig_height))
    axis.imshow(canvas, origin='upper')
    axis.axvline(qw - 0.5, color='white', linewidth=1.5, alpha=0.9)

    for index in order:
        qx, qy = q_points[index]
        gx, gy = g_points[index]
        color = plt.cm.viridis(color_norm(float(match_scores[index])))
        # The candidate is offset to the right side of the combined canvas.
        axis.plot(
            [qx, qw + gx], [qy, gy],
            color=color, linewidth=0.9, alpha=0.65, solid_capstyle='round'
        )
        axis.scatter([qx, qw + gx], [qy, gy], color=[color], s=16, zorder=3)

    axis.text(
        qw / 2, 8, f'Query\n{query_frame}',
        ha='center', va='top', color='white', fontsize=9,
        bbox=dict(facecolor='black', alpha=0.55, pad=3),
    )
    axis.text(
        qw + gw / 2, 8, f'Candidate\n{candidate_frame}',
        ha='center', va='top', color='white', fontsize=9,
        bbox=dict(facecolor='black', alpha=0.55, pad=3),
    )
    axis.set_xlim(-1, qw + gw)
    axis.set_ylim(max(qh, gh), -1)
    axis.axis('off')
    fig.suptitle(
        f'Actual matcher correspondences: {len(q_points)} total; '
        f'showing {len(order)} strongest',
        fontsize=14,
    )
    fig.tight_layout()

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=160, bbox_inches='tight')
        print('Saved:', save_path)
    display(fig)
    plt.close(fig)


def build_scorer_from_config():
    cache = RDD_CACHE_ROOT if SCORE_BACKEND == 'rdd' else LOMA_CACHE_ROOT
    checkpoint = RDD_CHECKPOINT if SCORE_BACKEND == 'rdd' else LOMA_CHECKPOINT
    return build_pair_scorer(SCORE_BACKEND, checkpoint, cache, DEVICE)

In [ ]:
def score_current_entries(entries, backend=SCORE_BACKEND):
    scorer = build_scorer_from_config() if backend not in (None, '', 'none') else None
    if scorer is None:
        print('No fresh scorer selected; grid titles will use stored mining scores when available.')
    else:
        print(f'Fresh scorer ready: {scorer.backend} on {scorer.device}')
    return scorer


def run_visualization(entries=None, scorer=None):
    if entries is None:
        entries = load_index(INDEX_PATH)
    issues = validate_entries(entries, DATASET_ROOT)
    print_entry_summary(entries[0])
    results = show_examples(
        entries,
        n_queries=N_QUERIES,
        n_positives=N_POSITIVES,
        n_negatives=N_NEGATIVES,
        selection_mode=SELECTION_MODE,
        seed=SEED,
        dataset_root=DATASET_ROOT,
        scorer=scorer,
    )
    return issues, results


if INDEX_PATH.exists():
    entries = load_index(INDEX_PATH)
    print(f'Loaded {len(entries):,} entries from {INDEX_PATH}')
    print_entry_summary(entries[0])
else:
    entries = []
    print(f'Index not found: {INDEX_PATH}')
    print('Edit the configuration cell, then run this cell again.')

## Typical usage

1. Set `DATASET_ROOT`, `INDEX_PATH`, and the cache/checkpoint paths in the configuration cell.
2. Run the loader and validation cells.
3. For image-only inspection, leave `SCORE_BACKEND = 'none'`.
4. To rescore the same pairs, set `SCORE_BACKEND = 'rdd'` or `'loma'`, choose the matching cache, and run:

```python
scorer = score_current_entries(entries)
issues, results = run_visualization(entries, scorer)
```

5. To display actual matcher correspondences for a selected pair:

```python
show_match_overlay(
    scorer,
    entries[0]['query_frame'],
    entries[0]['positives'][0]['frame'],
    save_path=OUTPUT_ROOT / 'example_overlay.png',
)
```

For raw mining shard files, point `INDEX_PATH` at a report such as `strong-matches_train_0.json`; the loader reads its `selected_frames` field and preserves stored mining scores when present.

`MINING_BACKEND` identifies which backend produced the pairs. With `rdd`, the
notebook prefers `indices/rdd/`; with `loma`, it uses `indices/loma/`. Set
`PAIR_INDEX` when inspecting a legacy shared index or a specific raw shard.
`SCORE_BACKEND` is independent: it controls optional fresh rescoring, while
`MINING_BACKEND` controls which mined index and cache are inspected.


In [ ]:
# 1. Inspect images only
# Leave SCORE_BACKEND = 'none' in the configuration cell.
entries = load_index(INDEX_PATH)
issues, results = run_visualization(entries)


In [ ]:
# 2. Inspect another split or index
SPLIT = 'test'
INDEX_PATH = _default_index_for_split(SPLIT)
entries = load_index(INDEX_PATH)
issues, results = run_visualization(entries)

# For strict CzechLynx validation, use:
# strong-matches_val_combined.json
#
# For the original Lynx dataset, update DATASET_ROOT and INDEX_PATH.


In [ ]:
# 3. Validate identities, self-matches, and missing images
entries = load_index(INDEX_PATH)
issues = validate_entries(entries, DATASET_ROOT, max_report=50)
issues[:10]


In [ ]:
# 4. Inspect cached feature coverage
missing_rdd, bad_rdd = cache_report(entries, RDD_CACHE_ROOT)
print('RDD missing:', len(missing_rdd), 'bad:', len(bad_rdd))
show_feature_info(entries[0], RDD_CACHE_ROOT)

missing_loma, bad_loma = cache_report(entries, LOMA_CACHE_ROOT)
print('LoMa missing:', len(missing_loma), 'bad:', len(bad_loma))
show_feature_info(entries[0], LOMA_CACHE_ROOT)

# Both cache schemas are supported:
# keypoints/descriptors/scores/image_size
# compact k/d/hw


In [ ]:
# 5. Choose candidate selection
results = show_examples(
    entries,
    n_queries=6,
    n_positives=3,
    n_negatives=3,
    selection_mode='mixed',  # random, strongest, weakest, or mixed
    seed=17,
    dataset_root=DATASET_ROOT,
)

# strongest and weakest use stored mining scores when available.
# Combined indices normally contain paths only, so use random or recompute scores.


In [ ]:
# 6. Recompute RDD/LightGlue scores
SCORE_BACKEND = 'rdd'
RDD_CACHE_ROOT = Path(
    '/shared/sets/datasets/vision/czechlynx/checkpoints/'
    'czechlynx-time-closed/rdd-cache'
)
RDD_CHECKPOINT = ''  # empty = pretrained RDD/LightGlue
scorer = build_scorer_from_config()
issues, results = run_visualization(entries, scorer)

# To score a fine-tuned LightGlue checkpoint, set for example:
# RDD_CHECKPOINT = (
#     '/shared/sets/datasets/vision/czechlynx/checkpoints/'
#     'czechlynx-time-closed/rdd-finetuned-legacy/epoch_299/model.safetensors'
# )
# scorer = build_scorer_from_config()
# issues, results = run_visualization(entries, scorer)


In [ ]:
# 7. Recompute LoMa scores
SCORE_BACKEND = 'loma'
LOMA_CACHE_ROOT = Path(
    '/shared/sets/datasets/vision/czechlynx/checkpoints/'
    'czechlynx-time-closed/loma-b-cache'
)
LOMA_CHECKPOINT = ''  # empty = pretrained LoMa-B
scorer = build_scorer_from_config()
issues, results = run_visualization(entries, scorer)

# To score a fine-tuned LoMa checkpoint bundle, set for example:
# LOMA_CHECKPOINT = (
#     '/shared/sets/datasets/vision/czechlynx/checkpoints/'
#     'czechlynx-time-closed/loma-b-finetuned-legacy/epoch_299'
# )
# scorer = build_scorer_from_config()
# issues, results = run_visualization(entries, scorer)


In [ ]:
# 8. Compare pretrained and fine-tuned checkpoints
query = entries[0]['query_frame']
positive = entries[0]['positives'][0]['frame']
negative = entries[0]['negatives'][0]['frame']

SCORE_BACKEND = 'loma'
LOMA_CHECKPOINT = ''
pretrained = build_scorer_from_config()

LOMA_CHECKPOINT = (
    '/shared/sets/datasets/vision/czechlynx/checkpoints/'
    'czechlynx-time-closed/loma-b-finetuned-trainval-4gpu/epoch_299'
)
finetuned = build_scorer_from_config()

for label, current_scorer in [
    ('pretrained', pretrained),
    ('fine-tuned', finetuned),
]:
    print(label)
    print(' positive:', current_scorer.score(query, positive))
    print(' negative:', current_scorer.score(query, negative))

def pair_margin(current_scorer, query_frame, positive_frame, negative_frame):
    pos = current_scorer.score(query_frame, positive_frame)
    neg = current_scorer.score(query_frame, negative_frame)
    return {
        'positive_score': pos.get('score'),
        'negative_score': neg.get('score'),
        'margin': pos.get('score', float('nan')) - neg.get('score', float('nan')),
    }

pair_margin(finetuned, query, positive, negative)


In [ ]:
# 9. Save figures
save_path = OUTPUT_ROOT / 'train_example_000.png'
show_pair_grid(
    entries[0],
    n_positives=2,
    n_negatives=2,
    selection_mode='mixed',
    seed=0,
    dataset_root=DATASET_ROOT,
    scorer=scorer,
    save_path=save_path,
)

# Save several examples.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
for index, entry in enumerate(entries[:5]):
    show_pair_grid(
        entry,
        n_positives=2,
        n_negatives=2,
        selection_mode='mixed',
        seed=SEED + index,
        dataset_root=DATASET_ROOT,
        scorer=scorer,
        save_path=OUTPUT_ROOT / f'pair_example_{index:03d}.png',
    )


In [ ]:
# 10a. Inspect available candidates for a query
entry_index = 15  # choose which query entry to inspect
entry = entries[entry_index]

print('Query:', entry['query_frame'])
print()
print('Positive candidates:')
for index, item in enumerate(entry['positives']):
    print(f'  [{index}] {item["frame"]}')

print('Negative candidates:')
for index, item in enumerate(entry['negatives']):
    print(f'  [{index}] {item["frame"]}')


In [ ]:
# 10b. Draw correspondences for the selected query and candidate
entry_index = 3
candidate_type = 'positives'  # choose 'positives' or 'negatives'
candidate_index = 3

entry = entries[entry_index]
query = entry['query_frame']
candidate = entry[candidate_type][candidate_index]['frame']

show_match_overlay(
    scorer,
    query,
    candidate,
    dataset_root=DATASET_ROOT,
    max_lines=10,
    save_path=OUTPUT_ROOT / (
        f'overlay_{entry_index}_{candidate_type}_{candidate_index}.png'
    ),
)


In [ ]:
# 11. Inspect a raw mining shard
# Raw reports preserve mining scores, unlike the usual combined index.
raw_index_root = (Path(_PAIR_INDEX_OVERRIDE).parent
                   if _PAIR_INDEX_OVERRIDE else
                   (MINING_INDEX_ROOT if (MINING_BACKEND == 'loma' or MINING_INDEX_ROOT.exists() or 'PAIR_MINING_BACKEND' in os.environ) else INDEX_ROOT))
INDEX_PATH = raw_index_root / 'strong-matches_train_0.json'
raw_entries = load_index(INDEX_PATH)
validate_entries(raw_entries, DATASET_ROOT)
show_examples(
    raw_entries,
    n_queries=3,
    n_positives=3,
    n_negatives=3,
    selection_mode='strongest',
    seed=SEED,
    dataset_root=DATASET_ROOT,
)

# Combined training indices are normally consumed by fine-tuning.
# Raw shards are useful for inspecting the original mining scores.


In [ ]:
# 11. Configure pair-contribution diagnostics
# The defaults scan only a small sample. Set *_MAX_* to 0 to scan everything,
# but rescoring every pair can be expensive on a full CzechLynx index.
DIAGNOSTIC_MAX_ENTRIES = 20
DIAGNOSTIC_MAX_POSITIVES = 0       # 0 = all positives in each entry
DIAGNOSTIC_MAX_NEGATIVES = 5      # 0 = all negatives in each entry
DIAGNOSTIC_MARGIN = 0.5
DIAGNOSTIC_WEAK_SCORE = 0.15       # diagnostic threshold, not a training rule
DIAGNOSTIC_WEAK_MATCHES = 16      # diagnostic threshold, not a training rule
DIAGNOSTIC_OUTPUT = OUTPUT_ROOT / 'pair_contribution_diagnostics.csv'


def _diagnostic_limit(items, limit):
    items = list(items)
    return items if not limit else items[:int(limit)]


def _diagnostic_pair_score(current_scorer, query_frame, candidate_frame):
    result = dict(current_scorer.score(query_frame, candidate_frame))
    result.setdefault('score', float('nan'))
    result['score'] = float(result['score'])
    result.setdefault('matches', None)

    # LoMa's normal score helper does not return match counts. For diagnostics,
    # obtain the final filtered correspondence count from the actual matcher.
    if result['matches'] is None:
        try:
            q_points, g_points, match_scores = _raw_correspondences(
                current_scorer, query_frame, candidate_frame
            )
            result['matches'] = int(len(q_points))
            result['match_count_source'] = 'filtered matcher correspondences'
        except Exception as exc:
            result['matches_error'] = str(exc)
    if result.get('matches') is not None:
        result['matches'] = int(result['matches'])
    return result


def collect_pair_contribution_diagnostics(
    index_entries,
    current_scorer,
    max_entries=DIAGNOSTIC_MAX_ENTRIES,
    max_positives=DIAGNOSTIC_MAX_POSITIVES,
    max_negatives=DIAGNOSTIC_MAX_NEGATIVES,
    margin=DIAGNOSTIC_MARGIN,
    weak_score=DIAGNOSTIC_WEAK_SCORE,
    weak_matches=DIAGNOSTIC_WEAK_MATCHES,
):
    if current_scorer is None:
        raise ValueError(
            "Build an RDD or LoMa scorer first: set SCORE_BACKEND and run "
            "the scorer setup cell."
        )

    selected_entries = list(index_entries)
    if max_entries:
        selected_entries = selected_entries[:int(max_entries)]

    rows = []
    for entry_index, entry in enumerate(selected_entries):
        query_frame = entry['query_frame']
        positives = _diagnostic_limit(entry.get('positives', []), max_positives)
        negatives = _diagnostic_limit(entry.get('negatives', []), max_negatives)

        negative_scores = []
        for negative in negatives:
            try:
                negative_result = _diagnostic_pair_score(
                    current_scorer, query_frame, negative['frame']
                )
                score = negative_result.get('score', float('nan'))
                if np.isfinite(score):
                    negative_scores.append(float(score))
            except Exception:
                continue
        best_negative_score = max(negative_scores) if negative_scores else float('nan')

        for positive_index, positive in enumerate(positives):
            positive_frame = positive['frame']
            try:
                positive_result = _diagnostic_pair_score(
                    current_scorer, query_frame, positive_frame
                )
                positive_score = float(
                    positive_result.get('score', float('nan'))
                )
                positive_matches = positive_result.get('matches')
                positive_error = positive_result.get('error')
                if positive_result.get('matches_error'):
                    positive_error = positive_result['matches_error']
            except Exception as exc:
                positive_result = {}
                positive_score = float('nan')
                positive_matches = None
                positive_error = str(exc)

            if np.isfinite(positive_score) and np.isfinite(best_negative_score):
                score_margin = positive_score - best_negative_score
                loss = max(0.0, margin - positive_score + best_negative_score)
            else:
                score_margin = float('nan')
                loss = float('nan')

            if positive_error:
                status = 'scoring_error'
                contributes_to_rdd_loss = False
            elif positive_matches == 0:
                # This is the exact no-positive-match case for RDD/LightGlue:
                # the current RDD loss drops the positive pair.
                status = 'zero_filtered_matches'
                contributes_to_rdd_loss = False
            elif np.isfinite(loss) and loss > 0:
                if (
                    positive_score < weak_score
                    or positive_matches < weak_matches
                ):
                    status = 'weak_positive_active_loss'
                else:
                    status = 'active_loss'
                contributes_to_rdd_loss = True
            elif np.isfinite(loss):
                status = 'already_past_margin'
                contributes_to_rdd_loss = False
            else:
                status = 'missing_negative_score'
                contributes_to_rdd_loss = False

            query_meta = path_metadata(query_frame)
            positive_meta = path_metadata(positive_frame)
            rows.append({
                'entry_index': entry_index,
                'positive_index': positive_index,
                'query_frame': query_frame,
                'positive_frame': positive_frame,
                'query_identity': query_meta['identity'],
                'positive_identity': positive_meta['identity'],
                'query_source': query_meta['source'],
                'positive_source': positive_meta['source'],
                'positive_score': positive_score,
                'positive_matches': positive_matches,
                'best_negative_score': best_negative_score,
                'score_margin': score_margin,
                'margin_loss': loss,
                'status': status,
                'contributes_to_rdd_loss': contributes_to_rdd_loss,
                'backend': current_scorer.backend,
                'negative_candidates_scored': len(negatives),
                'error': positive_error or '',
            })
    return rows


In [ ]:
# 12. Run pair-contribution diagnostics
# Set SCORE_BACKEND = 'rdd' or 'loma', configure its cache/checkpoint, and
# run the corresponding scorer setup cell before running this cell.
if SCORE_BACKEND in (None, '', 'none'):
    raise RuntimeError(
        "Set SCORE_BACKEND to 'rdd' or 'loma' before running diagnostics."
    )

if 'scorer' not in globals() or scorer is None:
    scorer = build_scorer_from_config()

pair_diagnostics = collect_pair_contribution_diagnostics(
    entries,
    scorer,
)
print(f'Analyzed {len(pair_diagnostics):,} query-positive pairs')


In [ ]:
# 13. Rank and summarize suspicious pairs
from collections import Counter

def diagnostic_summary(rows):
    counts = Counter(row['status'] for row in rows)
    print('Status counts:')
    for status, count in counts.most_common():
        print(f'  {status}: {count:,}')

    valid_matches = [
        row['positive_matches'] for row in rows
        if isinstance(row['positive_matches'], int)
    ]
    valid_scores = [
        row['positive_score'] for row in rows
        if np.isfinite(row['positive_score'])
    ]
    if valid_matches:
        print(f'Positive matches: min={min(valid_matches)}, '
              f'median={np.median(valid_matches):.1f}, '
              f'max={max(valid_matches)}')
    if valid_scores:
        print(f'Positive scores: min={min(valid_scores):.4f}, '
              f'median={np.median(valid_scores):.4f}, '
              f'max={max(valid_scores):.4f}')
    return counts


def rank_diagnostic_rows(rows):
    priority = {
        'zero_filtered_matches': 0,
        'scoring_error': 1,
        'weak_positive_active_loss': 2,
        'active_loss': 3,
        'missing_negative_score': 4,
        'already_past_margin': 5,
    }
    return sorted(
        rows,
        key=lambda row: (
            priority.get(row['status'], 99),
            row['positive_matches']
            if isinstance(row['positive_matches'], int) else float('inf'),
            row['positive_score']
            if np.isfinite(row['positive_score']) else float('inf'),
        ),
    )


def print_diagnostic_rows(rows, limit=20):
    for row in rank_diagnostic_rows(rows)[:limit]:
        print(
            f"[{row['status']}] "
            f"matches={row['positive_matches']} "
            f"pos={row['positive_score']:.4f} "
            f"best_neg={row['best_negative_score']:.4f} "
            f"margin={row['score_margin']:.4f}"
        )
        print(f"  query:    {row['query_frame']}")
        print(f"  positive: {row['positive_frame']}")


diagnostic_summary(pair_diagnostics)
print()
print_diagnostic_rows(pair_diagnostics, limit=20)


In [ ]:
# 14. Visualize the worst pairs
# Zero-filtered-match pairs are the clearest RDD limitation examples.
# For LoMa, zero final filtered matches are a quality warning, but do not
# prove that the differentiable training score had zero gradient.
worst_rows = [
    row for row in rank_diagnostic_rows(pair_diagnostics)
    if row['status'] in (
        'zero_filtered_matches',
        'weak_positive_active_loss',
        'scoring_error',
    )
]

for number, row in enumerate(worst_rows[:6]):
    print(
        f"Example {number + 1}: {row['status']} | "
        f"matches={row['positive_matches']} | "
        f"positive_score={row['positive_score']:.4f}"
    )
    if row['status'] == 'scoring_error':
        print('  error:', row['error'])
        continue
    show_match_overlay(
        scorer,
        row['query_frame'],
        row['positive_frame'],
        dataset_root=DATASET_ROOT,
        max_lines=40,
        save_path=OUTPUT_ROOT / f'bad_pair_{number:03d}.png',
    )


In [ ]:
# 15. Export the diagnostics for paper/example selection
import csv

if pair_diagnostics:
    fields = list(pair_diagnostics[0].keys())
    DIAGNOSTIC_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with DIAGNOSTIC_OUTPUT.open('w', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(pair_diagnostics)
    print('Saved:', DIAGNOSTIC_OUTPUT)
else:
    print('No diagnostic rows to export.')
